[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/03_advanced_architectures/gans.ipynb)


# 04 . Generative Adversarial Networks (GANs)

## Concept — GANs (revision notes)

**What it is**
- Two networks play a game: a Generator G maps random noise z to fake samples; a Discriminator D outputs the probability that a sample is real.
- D is trained to tell real from fake; G is trained to fool D. Both improve against each other; no explicit likelihood is ever computed.

**Why it matters**
- GANs produce sharp samples with a single forward pass and launched modern image synthesis (StyleGAN, pix2pix, super-resolution).
- They are also the best-known example of *unstable* training, so they teach practical lessons about balancing two optimisers.

**When to use**
- When you want fast, sharp sample generation and can tolerate fiddly training; adversarial losses also appear as add-ons (perceptual realism terms).
- For stable training and likelihoods, VAEs or diffusion models are usually easier (see vaes.ipynb and diffusion_models.ipynb).

**Math & intuition**
- Original minimax objective: `min_G max_D  E[log D(x)] + E[log(1 - D(G(z)))]`. For an optimal D the objective equals a Jensen-Shannon divergence between real and generated distributions.
- In practice G uses the *non-saturating* loss `-log D(G(z))`: when D confidently rejects fakes, `log(1-D)` has almost no gradient, `-log D` has a strong one (section 5).
- At the theoretical equilibrium `p_G = p_data` and `D(x) = 0.5` everywhere.
- Mode collapse: G finds a few outputs that fool D and stops covering the rest of the data distribution.

### 1. A 1D toy distribution: two Gaussians

**What this cell does (plain language):** the real data is a mixture of two bumps (at -2 and +2). Having a bimodal target is the simplest setting where a generator can cover both modes or only one. We sample it and print basic statistics.

In [1]:
import math, time, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
torch.manual_seed(0)

def real_1d(n):
    comp = torch.randint(0, 2, (n, 1))                                    # which bump
    return torch.where(comp == 0, -2 + 0.4 * torch.randn(n, 1), 2 + 0.4 * torch.randn(n, 1))

x = real_1d(5000)
print("mean %.2f std %.2f | fraction > 0: %.2f" % (x.mean(), x.std(), (x > 0).float().mean()))


mean 0.03 std 2.04 | fraction > 0: 0.51


### 2. Generator and discriminator networks

**What this cell does (plain language):** both are small MLPs. G takes 4-dim noise and outputs one number; D takes a number and outputs a LOGIT (we use `BCEWithLogitsLoss`, which folds the sigmoid in for numerical stability). We print the shapes and parameter counts.

In [2]:
def make_G(zdim=4, out=1, h=32): return nn.Sequential(nn.Linear(zdim, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, out))
def make_D(inp=1, h=32): return nn.Sequential(nn.Linear(inp, h), nn.LeakyReLU(0.2), nn.Linear(h, h), nn.LeakyReLU(0.2), nn.Linear(h, 1))   # logit out
G, D = make_G(), make_D()
z = torch.randn(8, 4)
print("G(z):", G(z).shape, "| D(x) logits:", D(G(z)).shape, "| params G/D:", sum(p.numel() for p in G.parameters()), sum(p.numel() for p in D.parameters()))


G(z): torch.Size([8, 1]) | D(x) logits: torch.Size([8, 1]) | params G/D: 1249 1153


### 3. One adversarial step, carefully

**What this cell does (plain language):** a GAN step has two optimisation sub-steps. (1) D step: push D(real) toward 1 and D(fake) toward 0 -- with the fake batch DETACHED so no gradient reaches G. (2) G step: make D(G(z)) look real using the non-saturating loss (labels = 1 for fakes). We wrap this in a function and use `Adam(lr=2e-3, betas=(0.5, 0.999))`, the usual GAN setting.

In [3]:
bce = nn.BCEWithLogitsLoss()
def gan_step(G, D, optG, optD, real, zdim, bs):
    ones, zeros = torch.ones(bs, 1), torch.zeros(bs, 1)
    # ---- D step ----
    fake = G(torch.randn(bs, zdim)).detach()           # detach: do not backprop into G here
    lossD = bce(D(real), ones) + bce(D(fake), zeros)
    optD.zero_grad(); lossD.backward(); optD.step()
    # ---- G step ----
    fake = G(torch.randn(bs, zdim))                    # fresh, non-detached samples
    lossG = bce(D(fake), ones)                         # non-saturating: pretend fakes are real
    optG.zero_grad(); lossG.backward(); optG.step()
    return lossD.item(), lossG.item()

def train_1d(steps=1500, bs=128, seed=0, lr_g=2e-3, lr_d=2e-3):
    torch.manual_seed(seed); G, D = make_G(), make_D()
    oG = torch.optim.Adam(G.parameters(), lr=lr_g, betas=(0.5, 0.999)); oD = torch.optim.Adam(D.parameters(), lr=lr_d, betas=(0.5, 0.999))
    hist = []
    for s in range(steps):
        hist.append(gan_step(G, D, oG, oD, real_1d(bs), 4, bs))
    return G, D, hist
t0 = time.time(); G, D, hist = train_1d()
for s in [0, 300, 700, 1499]: print(f"step {s:4d} | D loss {hist[s][0]:.3f} | G loss {hist[s][1]:.3f}")
print(f"time {time.time()-t0:.1f}s")


step    0 | D loss 1.409 | G loss 0.661
step  300 | D loss 0.869 | G loss 1.176
step  700 | D loss 1.386 | G loss 0.631
step 1499 | D loss 1.385 | G loss 0.708
time 6.0s


### 4. Did G learn both modes?

**What this cell does (plain language):** we draw 5000 samples from the trained generator, compare the fraction in each mode and the overall mean/std to the real data, and plot both histograms.

In [4]:
with torch.no_grad(): fake = G(torch.randn(5000, 4))
real = real_1d(5000)
print("real: mean %.2f std %.2f | frac>0 %.2f" % (real.mean(), real.std(), (real > 0).float().mean()))
print("fake: mean %.2f std %.2f | frac>0 %.2f" % (fake.mean(), fake.std(), (fake > 0).float().mean()))
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.hist(real.numpy(), bins=60, alpha=0.5, density=True, label="real"); ax.hist(fake.numpy(), bins=60, alpha=0.5, density=True, label="generated")
ax.legend(); ax.set_title("1D GAN: real vs generated"); plt.tight_layout(); plt.show()


real: mean -0.01 std 2.04 | frac>0 0.50
fake: mean 0.31 std 1.89 | frac>0 0.52


### 5. Why the non-saturating loss: gradient comparison

**What this cell does (plain language):** early in training D easily spots fakes, so D(G(z)) is near 0. We compute the gradient of both G losses with respect to the discriminator logit `s` (D(G(z)) = sigmoid(s)) at a very negative `s`: the original `log(1 - sigmoid(s))` gives a tiny gradient, while `-log sigmoid(s)` gives a gradient of magnitude ~1.

In [5]:
for s_val in [-8.0, -4.0, 0.0, 4.0]:
    s = torch.tensor(s_val, requires_grad=True)
    sat = torch.log(1 - torch.sigmoid(s))               # minimise this (original minimax generator loss)
    g_sat, = torch.autograd.grad(sat, s)
    s2 = torch.tensor(s_val, requires_grad=True)
    nonsat = -F.logsigmoid(s2)                           # non-saturating: minimise -log D(G(z))
    g_non, = torch.autograd.grad(nonsat, s2)
    print(f"logit {s_val:5.1f} | grad of original loss {g_sat.item():+.4f} | grad of non-saturating {g_non.item():+.4f}")


logit  -8.0 | grad of original loss -0.0003 | grad of non-saturating -0.9997
logit  -4.0 | grad of original loss -0.0180 | grad of non-saturating -0.9820
logit   0.0 | grad of original loss -0.5000 | grad of non-saturating -0.5000
logit   4.0 | grad of original loss -0.9820 | grad of non-saturating -0.0180


### 6. Reading the loss curves (they are not a quality metric)

**What this cell does (plain language):** unlike supervised training, GAN losses do not decrease to a number that means 'good'. We plot D and G losses and the mean D output on real and fake data. In a healthy run D's loss hovers near `2*ln2 = 1.386` and D(x) values sit near 0.5 rather than at 0 or 1.

In [6]:
Dl = torch.tensor([h[0] for h in hist]); Gl = torch.tensor([h[1] for h in hist])
with torch.no_grad():
    d_real = torch.sigmoid(D(real_1d(2000))).mean().item(); d_fake = torch.sigmoid(D(G(torch.randn(2000, 4)))).mean().item()
print(f"mean D loss last 200 steps: {Dl[-200:].mean():.3f} (equilibrium 2ln2 = {2*math.log(2):.3f}) | G loss: {Gl[-200:].mean():.3f}")
print(f"D(real) = {d_real:.3f} | D(fake) = {d_fake:.3f}  (0.5/0.5 would be a perfectly fooled D)")
sm = lambda t: t.unfold(0, 50, 50).mean(1)
fig, ax = plt.subplots(figsize=(5.5, 3)); ax.plot(sm(Dl), label="D loss"); ax.plot(sm(Gl), label="G loss"); ax.legend(); ax.set_xlabel("x50 steps")
plt.tight_layout(); plt.show()


mean D loss last 200 steps: 1.386 (equilibrium 2ln2 = 1.386) | G loss: 0.704
D(real) = 0.495 | D(fake) = 0.491  (0.5/0.5 would be a perfectly fooled D)


### 7. 2D target: a ring of 8 Gaussians

**What this cell does (plain language):** a classic GAN stress test: 8 clusters evenly spaced on a circle of radius 2. We define the sampler plus a mode-coverage metric: how many of the 8 clusters receive at least 1% of the generated samples, and what fraction of samples land within 0.5 of some cluster centre (sample quality).

In [7]:
angles = torch.arange(8) * (2 * math.pi / 8)
centers = 2.0 * torch.stack([angles.cos(), angles.sin()], 1)           # (8,2)
def real_2d(n, g=None):
    k = torch.randint(0, 8, (n,)); return centers[k] + 0.1 * torch.randn(n, 2)

def mode_stats(samples):
    d = torch.cdist(samples, centers)                                  # (n,8) distance to each centre
    near, idx = d.min(1)
    good = near < 0.5                                                  # sample landed on some mode
    counts = torch.bincount(idx[good], minlength=8)
    return int((counts >= 0.01 * len(samples)).sum()), good.float().mean().item(), counts
m, q, c = mode_stats(real_2d(2000))
print("real data -> modes covered:", m, "| fraction near a mode:", round(q, 3), "| counts:", c.tolist())


real data -> modes covered: 8 | fraction near a mode: 1.0 | counts: [259, 252, 215, 266, 264, 262, 240, 242]


### 8. Train the 2D GAN and track mode coverage

**What this cell does (plain language):** we train a slightly bigger G/D pair on the ring and log mode coverage every few hundred steps. Coverage typically starts low (G dumps samples near the origin or on one cluster) and grows as the game progresses, though it may fluctuate -- GAN training is not monotonic.

In [8]:
def make_G2(): return make_G(zdim=8, out=2, h=64)
def make_D2(): return make_D(inp=2, h=64)
def train_2d(steps=2000, bs=256, seed=0, lr_g=1e-3, lr_d=1e-3, d_steps=1, log=True):
    torch.manual_seed(seed); G2, D2 = make_G2(), make_D2()
    oG = torch.optim.Adam(G2.parameters(), lr=lr_g, betas=(0.5, 0.999)); oD = torch.optim.Adam(D2.parameters(), lr=lr_d, betas=(0.5, 0.999))
    for s in range(steps):
        for _ in range(d_steps - 1):                                    # extra D-only updates if requested
            f = G2(torch.randn(bs, 8)).detach(); l = bce(D2(real_2d(bs)), torch.ones(bs, 1)) + bce(D2(f), torch.zeros(bs, 1)); oD.zero_grad(); l.backward(); oD.step()
        gan_step(G2, D2, oG, oD, real_2d(bs), 8, bs)
        if log and (s + 1) % 500 == 0:
            with torch.no_grad(): m, q, _ = mode_stats(G2(torch.randn(2000, 8)))
            print(f"step {s+1:4d} | modes covered {m}/8 | fraction near a mode {q:.2f}")
    return G2, D2
t0 = time.time(); G2, D2 = train_2d(); print(f"time {time.time()-t0:.1f}s")


step  500 | modes covered 8/8 | fraction near a mode 0.25


step 1000 | modes covered 8/8 | fraction near a mode 0.77


step 1500 | modes covered 8/8 | fraction near a mode 0.87


step 2000 | modes covered 8/8 | fraction near a mode 0.92
time 13.0s


### 9. Visualise real vs generated 2D samples

**What this cell does (plain language):** scatter plots of real points and generated points together make coverage obvious: you should see whether generated dots (orange) sit on the eight real clusters (blue) or leave some uncovered.

In [9]:
with torch.no_grad(): f = G2(torch.randn(1500, 8))
r = real_2d(1500)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(r[:, 0], r[:, 1], s=4, alpha=0.4, label="real"); ax.scatter(f[:, 0], f[:, 1], s=4, alpha=0.4, label="generated")
ax.set_aspect("equal"); ax.legend(markerscale=3); ax.set_title("8-Gaussian ring"); plt.tight_layout(); plt.show()


### 10. Mode collapse and imbalance experiments

**What this cell does (plain language):** mode collapse is G concentrating on a few modes. Whether a tiny run collapses depends on seed and balance, so rather than assume it we run several configurations (default; a very weak D; a very strong D with 5 updates per G step) and print the measured coverage and quality. Read the numbers: they show how sensitive GAN training is to the D/G balance.

In [10]:
configs = {"default (1 D step)": dict(), "weak D (lr_d=1e-4)": dict(lr_d=1e-4), "strong D (5 D steps)": dict(d_steps=5)}
for name, kw in configs.items():
    t0 = time.time(); Gx, _ = train_2d(steps=1000, log=False, **kw)
    with torch.no_grad(): m, q, cnt = mode_stats(Gx(torch.randn(2000, 8)))
    print(f"{name:22s} | modes covered {m}/8 | near-mode fraction {q:.2f} | per-mode counts {cnt.tolist()} | {time.time()-t0:.1f}s")
print("Mitigations in practice: WGAN-GP/spectral norm, minibatch discrimination, unrolled GANs, more diverse D inputs, tuning the D:G update ratio.")


default (1 D step)     | modes covered 8/8 | near-mode fraction 0.73 | per-mode counts [167, 166, 173, 186, 240, 214, 190, 122] | 7.4s


weak D (lr_d=1e-4)     | modes covered 1/8 | near-mode fraction 0.30 | per-mode counts [0, 0, 603, 0, 0, 0, 0, 0] | 7.1s


strong D (5 D steps)   | modes covered 2/8 | near-mode fraction 0.96 | per-mode counts [0, 2, 1833, 77, 9, 0, 0, 0] | 17.8s
Mitigations in practice: WGAN-GP/spectral norm, minibatch discrimination, unrolled GANs, more diverse D inputs, tuning the D:G update ratio.


## Common bugs

- **Not detaching fakes in the D step** — G accumulates gradients from D's loss and wastes compute (and can be updated wrongly if the optimizer steps). Fix: `fake = G(z).detach()` for D's update.
- **Using the saturating generator loss** — `log(1 - D(G(z)))` gives almost no gradient when D wins early. Fix: train G on `-log D(G(z))` (BCE with label 1).
- **Sigmoid in D plus `BCEWithLogitsLoss`** — double sigmoid squashes outputs and breaks learning. Fix: D outputs logits when using `BCEWithLogitsLoss`, or use sigmoid + `BCELoss`, not both.
- **Reading the loss as progress** — GAN losses oscillate and a low G loss may mean D collapsed. Fix: track sample quality/coverage (and for images FID or visual grids), not just the loss.
- **D overpowers G** — D loss goes to 0 and G gets no useful gradient. Fix: lower D's lr, add noise/label smoothing, spectral norm, or fewer D steps.
- **Mode collapse** — generated samples cover only a few modes (check per-mode counts as in section 10). Fix: WGAN-GP, minibatch statistics, unrolling, or more G capacity.
- **Default Adam betas** — `beta1=0.9` makes GAN training oscillate more. Fix: `betas=(0.5, 0.999)` is the common stable choice.
- **Forgetting `torch.no_grad()` when sampling** — evaluation builds graphs and wastes memory. Fix: wrap sampling in `torch.no_grad()`.

## Exercises

**Exercise 1.** Add one-sided label smoothing (use 0.9 as the real label) in the D step of the 1D GAN and compare mean D(real) afterwards.

In [11]:
# Your attempt for Exercise 1 here


**Solution 1**

In [12]:
def train_1d_smooth(steps=1000, bs=128, smooth=0.9):
    torch.manual_seed(0); Gs, Ds = make_G(), make_D()
    oG = torch.optim.Adam(Gs.parameters(), 2e-3, betas=(0.5, 0.999)); oD = torch.optim.Adam(Ds.parameters(), 2e-3, betas=(0.5, 0.999))
    for _ in range(steps):
        real = real_1d(bs); fake = Gs(torch.randn(bs, 4)).detach()
        lD = bce(Ds(real), torch.full((bs, 1), smooth)) + bce(Ds(fake), torch.zeros(bs, 1)); oD.zero_grad(); lD.backward(); oD.step()
        lG = bce(Ds(Gs(torch.randn(bs, 4))), torch.ones(bs, 1)); oG.zero_grad(); lG.backward(); oG.step()
    with torch.no_grad(): return torch.sigmoid(Ds(real_1d(2000))).mean().item()
print("mean D(real) with smoothing:", round(train_1d_smooth(), 3), "| without:", round(d_real, 3))


mean D(real) with smoothing: 0.457 | without: 0.495


**Exercise 2.** Prove the `detach` bug empirically: run a D-step loss WITHOUT detaching and show that G's parameters receive non-zero gradients.

In [13]:
# Your attempt for Exercise 2 here


**Solution 2**

In [14]:
G_t, D_t = make_G(), make_D()
fake = G_t(torch.randn(16, 4))                       # no detach
loss = bce(D_t(fake), torch.zeros(16, 1)); loss.backward()
print("G grad norm without detach:", round(sum(p.grad.norm() for p in G_t.parameters()).item(), 4))
G_t.zero_grad(); fake = G_t(torch.randn(16, 4)).detach()
print("with detach, G has grads:", any(p.grad is not None and p.grad.abs().sum() > 0 for p in G_t.parameters()))


G grad norm without detach: 0.0079
with detach, G has grads: False


**Exercise 3.** Implement the WGAN-GP gradient penalty function `gradient_penalty(D, real, fake)` and evaluate it on a batch of the 2D data.

In [15]:
# Your attempt for Exercise 3 here


**Solution 3**

In [16]:
def gradient_penalty(D, real, fake):
    eps = torch.rand(real.size(0), 1)
    xh = (eps * real + (1 - eps) * fake).requires_grad_(True)             # random points between real and fake
    g, = torch.autograd.grad(D(xh).sum(), xh, create_graph=True)
    return ((g.norm(dim=1) - 1) ** 2).mean()                             # push gradient norm toward 1 (1-Lipschitz critic)
with torch.no_grad(): fk = G2(torch.randn(128, 8))
print("GP value:", round(gradient_penalty(D2, real_2d(128), fk).item(), 4))


GP value: 7.2238


**Exercise 4.** Change the 1D target to THREE modes (-3, 0, 3) and report the fraction of samples near each mode after training (fresh run).

In [17]:
# Your attempt for Exercise 4 here


**Solution 4**

In [18]:
def real_3(n):
    c = torch.randint(0, 3, (n, 1)).float() * 3 - 3
    return c + 0.3 * torch.randn(n, 1)
torch.manual_seed(0); G3, D3 = make_G(), make_D()
oG = torch.optim.Adam(G3.parameters(), 2e-3, betas=(0.5, 0.999)); oD = torch.optim.Adam(D3.parameters(), 2e-3, betas=(0.5, 0.999))
for _ in range(1500): gan_step(G3, D3, oG, oD, real_3(128), 4, 128)
with torch.no_grad(): s = G3(torch.randn(3000, 4))
print("fraction near -3, 0, 3:", [round(((s - m).abs() < 1).float().mean().item(), 2) for m in (-3, 0, 3)])


fraction near -3, 0, 3: [0.38, 0.29, 0.29]


**Exercise 5.** Make the generator input noise dimension 1 instead of 4 and see how the 2D ring coverage changes after 1000 steps.

In [19]:
# Your attempt for Exercise 5 here


**Solution 5**

In [20]:
def make_G_small(): return make_G(zdim=1, out=2, h=64)
torch.manual_seed(0); Gs, Ds = make_G_small(), make_D2()
oG = torch.optim.Adam(Gs.parameters(), 1e-3, betas=(0.5, 0.999)); oD = torch.optim.Adam(Ds.parameters(), 1e-3, betas=(0.5, 0.999))
for _ in range(1000): gan_step(Gs, Ds, oG, oD, real_2d(256), 1, 256)
with torch.no_grad(): print("modes covered / near-mode fraction:", mode_stats(Gs(torch.randn(2000, 1)))[:2])


modes covered / near-mode fraction: (7, 0.6324999928474426)


**Exercise 6.** Use the trained 1D discriminator `D` to print D(x) for x in [-4,-2,0,2,4]. Where is it most confident the sample is real?

In [21]:
# Your attempt for Exercise 6 here


**Solution 6**

In [22]:
xs = torch.tensor([[-4.], [-2.], [0.], [2.], [4.]])
with torch.no_grad(): print({float(x): round(p.item(), 3) for x, p in zip(xs, torch.sigmoid(D(xs)))})


{-4.0: 0.435, -2.0: 0.519, 0.0: 0.201, 2.0: 0.473, 4.0: 0.397}
